# Survey Data Cleaning and Quality Assessment

## Overview
This notebook performs preliminary data cleaning and quality checks on survey responses. 
It includes:
- **Data filtering**: Removes responses from non-consenting participants and those without required language proficiency
- **Bot detection**: Identifies and analyzes potential bot/fraudulent responses using Qualtrics quality metrics
- **Data quality checks**: Examines duplicates, response duration, and demographic composition
- **Final output**: Produces a clean dataset ready for downstream analysis

To reproduce our filtering process, start from `survey_full.tsv`

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportion_confint
import seaborn as sns
import matplotlib.pyplot as plt

# Configure visualization
sns.set_theme("notebook", style="whitegrid", font_scale=1.3)

In [ ]:
# Data Loading
notebook_dir = Path(".").resolve()
project_root = notebook_dir.parent
data_file = project_root / "survey_full.tsv"

In [ ]:
df = pd.read_csv(data_file, sep="\t", skiprows=[1,2], encoding='utf-16')
df.head(3)

# Section 1: Initial Data Filters

Apply primary quality filters based on consent, language proficiency, and completeness of key survey items:
- **Q14**: Consent to participate in study
- **Q1**: Language proficiency threshold (must be B2 or higher)
- **Q43**: Respondents must have answered the chatbot question

In [ ]:
# q14: see how many respondents did not agree to take part in the survey
q14_summary = df["Q14"].value_counts(dropna=False).to_frame("Count")
q14_summary["Percentage"] = (q14_summary["Count"] / len(df) * 100).round(2)
print(q14_summary)

In [ ]:
# q1: see how many respondents do not have threshold language proficiency
q1_summary = df["Q1"].value_counts(dropna=True).to_frame("Count")
q1_summary["Percentage"] = (q1_summary["Count"] / len(df) * 100).round(2)
print(q1_summary)

In [ ]:
# Apply filters (keep responses that DON'T meet exclusion criteria)
df_clean = df[
    (df["Q14"] != "No (NON voglio partecipare allo studio)") &
    (df["Q1"] != "Intermedia (B2) o inferiore")
].copy()

print(f"After initial filters:                {len(df_clean)} responses ({100*len(df_clean)/len(df):.1f}%)")

In [ ]:
# q43: see how many respondents did not answer the chatbot question
q43_na_count = df["Q43"].isna().sum()
q43_na_pct = (q43_na_count / len(df) * 100).round(2)
print(f"Q43 NA values: {q43_na_count} ({q43_na_pct}%)")

# Section 2: Data Quality Assessment

Investigate potential issues: bot responses, fraud flags, duplicates, and response duration

## 2.1 Bot Risk Assessment

**Metric**: Q_RecaptchaScore - Qualtrics Recaptcha bot probability score
- Range: 0.0 to 1.0 (higher = more likely to be bot)
- Threshold: Score ≥ 0.5 indicates likely bot behavior
- Reference: https://pmc.ncbi.nlm.nih.gov/articles/PMC10818231/#ref17

**Caveat**: this metric might have false positives.

### Demographic Comparison: Human vs Likely Bot Responses

To understand systematic differences between likely human and bot responses, we compare demographic characteristics across groups. This helps identify whether bots have different demographic profiles.

In [ ]:
# count n of responsed below human thresholds
count_03 = len(df_clean[df_clean["Q_RecaptchaScore"] <= 0.3])
count_05 = len(df_clean[df_clean["Q_RecaptchaScore"] < 0.5])
print("Count (0.3):", count_03)
print("Count (0.5):", count_05)

#### Define Demographic Mappings

Collapse fine-grained categories into broader groups for statistical comparison

In [ ]:
# more coarse-grained demographic groups
# Mapping education
education_map = {k: 'Graduates' for k in [
    'Laurea magistrale o master di primo livello',
    'Laurea triennale o a ciclo unico',
    'Dottorato di ricerca',
    'Master di secondo livello'
]}
education_map.update({k: 'Non-graduates' for k in [
    'Diploma di scuola superiore',
    'Istruzione secondaria di primo grado (medie)'
]})

# Mapping geography
def geography_map(x):
    if x in ['Nord-Ovest, Italia', 'Nord-Est, Italia']:
        return 'Nord'
    elif x in ['Sud Italia', 'Isole, Italia']:
        return 'Sud + Isole'
    elif x == 'Centro Italia':
        return 'Centro'
    elif x == 'Non vivo in Italia':
        return 'Non vivo in Italia'

# Mapping gender
def gender_map(x):
    if x == 'Uomo':
        return 'Uomo'
    elif x == 'Donna':
        return 'Donna'
    else:
        return 'Altro'

# Mapping income
def income_map(x):
    if x in ['Bassa', 'Medio Bassa']:
        return 'Lower'
    elif x in ['Medio Alta', 'Alta']:
        return 'Higher'
    elif x == 'Media':
        return 'Media'

def age_map(x):
    if x in ['18-24 anni', '25-34 anni']:
        return '18-34'
    elif x in ['35-44 anni', '45-54 anni']:
        return '35-54'
    else:
        return '55+'

In [ ]:
# Split into human and bot-flagged groups for comparison
human = df_clean[df_clean["Q_RecaptchaScore"] >= 0.5].copy()
bot = df_clean[df_clean["Q_RecaptchaScore"] < 0.5].copy()

# Apply demographic mappings (note: using group_df to avoid overwriting df_clean)
for group_df in [human, bot]:
    group_df['Education_mapped'] = group_df['Istruzione'].map(education_map)
    group_df['Geography_mapped'] = group_df['Q10'].apply(geography_map)
    group_df['Gender_mapped'] = group_df['Q17'].apply(gender_map)
    group_df['Income_mapped'] = group_df['Q16'].apply(income_map)
    group_df['Age_mapped'] = group_df['Q6'].apply(age_map)

# Dictionary mapping demographic names to their column names
demographics_dict = {
    'Gender': 'Gender_mapped',
    'Age': 'Age_mapped',
    'Geography': 'Geography_mapped',
    'Education': 'Education_mapped',
    'Educational Area': 'Q20',
    'Income': 'Income_mapped'
}

In [ ]:
def analyze_demographics(human_df, bot_df, demographics_dict):
    """
    Analyze demographic differences between two populations using chi-square test.
    
    Parameters:
    -----------
    human_df : pd.DataFrame
        DataFrame with human/reference group
    bot_df : pd.DataFrame
        DataFrame with bot/comparison group
    demographics_dict : dict
        Mapping of demographic names to column names
        
    Returns:
    --------
    dict : Results with chi2, p-value, Cramér's V, effect size, and contingency table
    """
    results = {}

    for demo_name, column_name in demographics_dict.items():
        # Create aligned contingency table
        human_counts = human_df[column_name].value_counts().sort_index()
        bot_counts = bot_df[column_name].value_counts().sort_index()
        all_categories = sorted(set(human_counts.index) | set(bot_counts.index))

        contingency_table = pd.DataFrame({
            'Human': human_counts.reindex(all_categories, fill_value=0),
            'Bot': bot_counts.reindex(all_categories, fill_value=0)
        })

        # Statistical tests
        chi2, p_value, _, _ = chi2_contingency(contingency_table.values)
        n = contingency_table.sum().sum()
        cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))

        # Effect size interpretation
        effect_size = ("Large" if cramers_v >= 0.5 else
                      "Medium" if cramers_v >= 0.3 else
                      "Small" if cramers_v >= 0.1 else "Negligible")

        significance = ("***" if p_value < 0.001 else "**" if p_value < 0.01 else
                       "*" if p_value < 0.05 else "ns")

        results[demo_name] = {
            'chi2': chi2, 'p_value': p_value, 'cramers_v': cramers_v,
            'effect_size': effect_size, 'significance': significance,
            'contingency_table': contingency_table
        }

    return results

def summary_report(results, human_df, bot_df):
    """
    Create and print concise summary report of demographic analysis.
    
    Parameters:
    -----------
    results : dict
        Results from analyze_demographics()
    human_df, bot_df : pd.DataFrame
        DataFrames for reporting sample sizes
        
    Returns:
    --------
    pd.DataFrame : Summary table sorted by effect size
    """
    print(f"Sample Sizes: Human {len(human_df)} | Bot {len(bot_df)} | Ratio {len(human_df)/len(bot_df):.1f}:1")
    print("="*80)

    summary_df = pd.DataFrame([
        {'Demographic': demo, 'Chi2': r['chi2'], 'Cramers_V': r['cramers_v'],
         'Effect_Size': r['effect_size'], 'P_Value': r['p_value'], 'Significance': r['significance']}
        for demo, r in results.items()
    ]).sort_values('Cramers_V', ascending=False)

    print(summary_df.to_string(index=False, float_format='%.3f'))
    return summary_df

def plot_differences(results, figsize=(15, 10)):
    """
    Create side-by-side bar plots comparing demographic distributions.
    
    Parameters:
    -----------
    results : dict
        Results from analyze_demographics()
    figsize : tuple
        Figure size for matplotlib
    """
    fig, axes = plt.subplots(2, 3, figsize=figsize)
    axes = axes.flatten()

    for i, (demo_name, result) in enumerate(results.items()):
        if i < len(axes):
            contingency = result['contingency_table']
            human_pct = contingency['Human'] / contingency['Human'].sum() * 100
            bot_pct = contingency['Bot'] / contingency['Bot'].sum() * 100

            x = np.arange(len(contingency.index))
            axes[i].bar(x - 0.2, human_pct, 0.4, label='Human', alpha=0.8, color='skyblue')
            axes[i].bar(x + 0.2, bot_pct, 0.4, label='Bot', alpha=0.8, color='salmon')
            axes[i].set_title(f'{demo_name} (V={result["cramers_v"]:.3f})')
            axes[i].set_xticks(x)
            axes[i].set_xticklabels(contingency.index, rotation=45, ha='right')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Compare human and bot
results = analyze_demographics(human, bot, demographics_dict)
summary_report(results, human, bot)
plot_differences(results)

non significant, except for geography

In [ ]:
# Create duration in minutes
df_clean['duration_min'] = df_clean["Duration (in seconds)"] / 60

# Split groups
human_duration = df_clean.loc[df_clean["Q_RecaptchaScore"] >= 0.5, 'duration_min']
bot_duration = df_clean.loc[df_clean["Q_RecaptchaScore"] < 0.5, 'duration_min']

# Summary stats
print("Human duration (min):", human_duration.describe())
print("Bot duration (min):", bot_duration.describe())

# Normality test (Shapiro-Wilk)
# sample 184 for comparable size
print("Shapiro Human:", stats.shapiro(human_duration.sample(184, random_state=42)))
print("Shapiro Bot:", stats.shapiro(bot_duration.sample(min(184,len(bot_duration)), random_state=42)))
print(f"-----Median duration Human: {human_duration.median():.3f} minutes")
print(f"-----Median duration Bot: {bot_duration.median():.3f} minutes")


# Mann-Whitney U test (non-parametric)
u_stat, p_val = stats.mannwhitneyu(human_duration, bot_duration, alternative='two-sided')
print(f"Mann-Whitney U test: U={u_stat:.2f}, p={p_val:.4f}")

Even though the median duration is slightly lower for bots, this difference is not significant.

## 2.2 Fraud Detection

**Metric**: Q_RelevantIDFraudScore - Qualtrics fraud detection score
- Threshold: Score ≥ 0.30 (30%) indicates potential fraud
- Note: Uses IP-based and behavioral heuristics

In [ ]:
count_fraud = len(df_clean[df_clean["Q_RelevantIDFraudScore"] >= 0.3])
print("Count (0.3):", count_fraud)

## 2.3 Duplicate Response Detection

**Primary Metric**: Q_DuplicateRespondent
- Current standard for Qualtrics duplicate detection
- Replaces deprecated fields Q_RelevantIDDuplicate (deprecated July 2025)

**Analysis**: Check for duplicates among elderly respondents (65+) as special case

In [ ]:
# Count values in Q_DuplicateRespondent column
duplicate_counts = df_clean['Q_DuplicateRespondent'].value_counts()
print(duplicate_counts)

In [ ]:
# count how many are over the age of 65
count_65_plus = df_clean[(df_clean['Q_DuplicateRespondent'] == True) &
                         (df_clean['Q6'] == "Dai 65 anni in su")].shape[0]

print("Number of duplicate respondents aged 65+:", count_65_plus)

## 2.4 Response Duration Analysis

Survey response time can indicate engagement level or bot behavior.
- Very short responses may indicate insufficient engagement
- Very long responses may indicate inattention or survey resumption
- We use IQR-based outlier detection for robustness

In [ ]:
# Convert seconds to minutes
duration_min = df_clean["Duration (in seconds)"] / 60

In [ ]:
# plot response time distribution
bins = len(duration_min)
sns.histplot(duration_min, bins=bins)
plt.xlabel("Duration (minutes)")
plt.ylabel("Count")
plt.xlim(0, 100)

In [ ]:
# calculate plot time distribution
bins = [0, 1, 3, 7, 15, 20, 30, 60, 300, float('inf')]
labels = [
    "Below 1 min",
    "1 to 2.99 min",
    "3 to 6.99 min",
    "7 to 14.99 min",
    "15 to 20 min",
    "20 to 30 min",
    "30 to 60 min",
    "60 min - 5 hours",
    "Above 5 hours"

]

counts = pd.cut(duration_min, bins=bins, right=False, labels=labels).value_counts().sort_index()

print(f"Mean duration: {duration_min.mean():.2f} minutes")
print(f"Median duration: {duration_min.median():.2f} minutes")
print(f"Min duration: {duration_min.min():.2f} minutes")
print(f"Max duration: {duration_min.max():.2f} minutes")
print(counts)

**Interpretation**:
- Low duration associated with participants that did not use technology a lot and received restricted numbers of Qs
- Long duration (> 5 hours): Likely survey suspension/resumption rather than continuous engagement

In [ ]:
# find outliers based on qualtrics metrics: median ± 2×std
median_dur = duration_min.median()
std_dur = duration_min.std()
outliers = duration_min[(duration_min < median_dur - 2*std_dur) | (duration_min > median_dur + 2*std_dur)]

print(f"Median: {median_dur:.2f} min, Std: {std_dur:.2f} min, Outliers: {len(outliers)}")

In [ ]:
# find duration outliers with IQR-based filtering to be more robust to extreme values
Q1 = duration_min.quantile(0.25)
Q3 = duration_min.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

iqr_outliers = duration_min[(duration_min < lower) | (duration_min > upper)]

print(f"IQR lower: {lower:.2f} min, upper: {upper:.2f} min")
print(f"Outliers using IQR: {len(iqr_outliers)}")

# Section 3: Final Filtering and Clean Dataset Creation

This section applies all quality filters to produce the final production-ready dataset.

**Filtering stages:**
1. **Eligibility filters**: Consent and language proficiency requirements
2. **Quality filters**: Duplicate detection and response duration outliers
3. **Output**: Clean dataset summary with filtering statistics

In [ ]:
# Apply eligibility filters
original_count = len(df)

df_clean = df[
    (df["Q14"] != "No (NON voglio partecipare allo studio)") &
    (df["Q1"] != "Intermedia (B2) o inferiore")
].copy()

after_eligibility = len(df_clean)
eligibility_filtered = original_count - after_eligibility

print(f"Original rows: {original_count}")
print(f"Rows filtered for eligibility: {eligibility_filtered}")
print(f"Rows after eligibility filter: {after_eligibility}")
print()

initial_count = df_clean.shape[0]

# Calculate duration in minutes
duration_min = df_clean["Duration (in seconds)"] / 60

# Calculate IQR-based lower bound
Q1 = duration_min.quantile(0.25)
Q3 = duration_min.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR

# Duplicates filtering logic
mask_to_exclude = (
    (df_clean['Q_DuplicateRespondent'] == False) |
    ((df_clean['Q_DuplicateRespondent'] == True) & (df_clean['Q6'] == "Dai 65 anni in su")) |
    (df_clean['Q_RelevantIDDuplicate'] == True)
)

# Apply dynamic IQR-based duration filter (instead of hardcoded 2.59)
mask_to_exclude = mask_to_exclude & (duration_min >= lower_bound)

# Apply filters
df_clean = df_clean[~mask_to_exclude]

# Report results
final_count = df_clean.shape[0]
filtered_count = initial_count - final_count
print(f"IQR-based lower bound: {lower_bound:.2f} minutes")
print(f"Number of rows filtered (removed) by quality filters: {filtered_count}")
print(f"Number of rows remaining (kept): {final_count}")
print()

# Filter: Keep only respondents who answered chatbot question (Q43)
df_clean = df_clean[df_clean['Q43'].notna()].copy()
print(f"After filtering Q43 NA values: {len(df_clean)} rows")

In [ ]:
# df_clean_copy = df_clean.copy()
# for col in df_clean_copy.select_dtypes(include=['object']).columns:
#     df_clean_copy[col] = df_clean_copy[col].astype(str).str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False)

# output_file = project_root / 'survey_filtered.tsv'
# df_clean_copy.to_csv(output_file, sep="\t", index=False, lineterminator='\n')
# print(f"Cleaned dataset saved  ({len(df_clean_copy)} rows)")